# Text Analytics 2023-2024 - 6th mini-lab


## Topics Covered

* Introduction to Transformers and the HuggingFace🤗 library using Tensorflow and Keras

## **Install TensorFlow and HuggingFace🤗**

In [ ]:
%%capture
!pip install -U transformers
!pip install -U tensorflow

In [1]:
import transformers
import tensorflow
transformers.__version__

tensorflow.__version__

'2.15.0'

## **Download  & explore 20newsgroups dataset**

In [1]:
from sklearn.datasets import fetch_20newsgroups
twenty_train = fetch_20newsgroups(subset='train')  #, remove=('headers', 'footers', 'quotes'))
twenty_test = fetch_20newsgroups(subset='test')
print("Catergories")
print(twenty_train.target_names) #prints all the categories
print("-------------")
twenty_train.target_names
print("First dataset's sample")
print("\n".join(twenty_train.data[0].split("\n")))
print("------------")
print("First dataset's sample category: ",twenty_train.target[0])

Catergories
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
-------------
First dataset's sample
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this 

### **Split dataset into train (70%) & validation (30%)**

In [2]:
twenty_test.target

array([ 7,  5,  0, ...,  9,  6, 15])

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(twenty_train.data,
                                                  twenty_train.target,
                                                  test_size=0.3,
                                                  random_state=12547392)
X_test, y_test = twenty_test.data, twenty_test.target

type(X_train)

list

### **Convert labels to 1-hot vectors**

In [ ]:
import tensorflow as tf

y_train_1_hot = tf.keras.utils.to_categorical(y_train,
                                              num_classes=len(twenty_train.target_names))
y_val_1_hot = tf.keras.utils.to_categorical(y_val,
                                            num_classes=len(twenty_train.target_names))
y_test_1_hot = tf.keras.utils.to_categorical(y_test,
                                             num_classes=len(twenty_train.target_names))

for lidx,label in enumerate(twenty_train.target_names):
  print("Index: {} Category: {}".format(lidx,label))
print("Label index: {} | 1-hot vector:  {}".format(y_train[0],
                                                   y_train_1_hot[0]))
print("Label index: {} | 1-hot vector:  {}".format(y_train[10],
                                                   y_train_1_hot[10]))

Index: 0 Category: alt.atheism
Index: 1 Category: comp.graphics
Index: 2 Category: comp.os.ms-windows.misc
Index: 3 Category: comp.sys.ibm.pc.hardware
Index: 4 Category: comp.sys.mac.hardware
Index: 5 Category: comp.windows.x
Index: 6 Category: misc.forsale
Index: 7 Category: rec.autos
Index: 8 Category: rec.motorcycles
Index: 9 Category: rec.sport.baseball
Index: 10 Category: rec.sport.hockey
Index: 11 Category: sci.crypt
Index: 12 Category: sci.electronics
Index: 13 Category: sci.med
Index: 14 Category: sci.space
Index: 15 Category: soc.religion.christian
Index: 16 Category: talk.politics.guns
Index: 17 Category: talk.politics.mideast
Index: 18 Category: talk.politics.misc
Index: 19 Category: talk.religion.misc
Label index: 7 | 1-hot vector:  [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Label index: 9 | 1-hot vector:  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


### **Custom Keras callback for calculating f1, precision, recall at the end of each epoch**

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score
import os
import numpy as np

class Metrics(tf.keras.callbacks.Callback):
    def __init__(self, valid_data):
        super(Metrics, self).__init__()
        self.validation_data = valid_data

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_predict = np.argmax(tf.nn.softmax(self.model.predict(self.validation_data[0]).logits,
                                              axis=-1), -1)
        val_targ = self.validation_data[1]
        if len(val_targ.shape) == 2 and val_targ.shape[1] != 1:
            val_targ = np.argmax(val_targ, -1)
        val_targ = tf.cast(val_targ, dtype=tf.float32)


        _val_f1 = f1_score(val_targ, val_predict, average="weighted")
        _val_recall = recall_score(val_targ, val_predict, average="weighted")
        _val_precision = precision_score(val_targ, val_predict, average="weighted")

        logs['val_f1'] = _val_f1
        logs['val_recall'] = _val_recall
        logs['val_precision'] = _val_precision
        print(" — val_f1: %f — val_precision: %f — val_recall: %f" % (_val_f1, _val_precision, _val_recall))
        return

## **Text Classification with HuggingFace Transformers** 🤗

1. Out of the box classification with `BertForSequenceClassification`.

In [ ]:
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_clf = TFAutoModelForSequenceClassification.from_pretrained('bert-base-uncased',
                                        num_labels=len(twenty_train.target_names))
bert_clf.summary()

2024-03-11 22:13:37.712481: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:08:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-11 22:13:37.803263: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:08:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-11 22:13:37.803335: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:08:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-11 22:13:37.804042: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate

Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  15380     
                                                                 
Total params: 109,497,620
Trainable params: 109,497,620
Non-trainable params: 0
_________________________________________________________________


In [ ]:
def tokenize_text(data, tokenizer, max_length=250):
  return tokenizer(data, add_special_tokens=True, padding='max_length',
                   max_length=max_length, truncation=True, return_tensors='tf')

bert_train = tokenize_text(X_train, bert_tokenizer)
bert_val = tokenize_text(X_val, bert_tokenizer)
bert_test = tokenize_text(X_test, bert_tokenizer)

In [ ]:
bert_train

{'input_ids': <tf.Tensor: shape=(7919, 250), dtype=int32, numpy=
array([[ 101, 2013, 1024, ...,    0,    0,    0],
       [ 101, 2013, 1024, ..., 2013, 2068,  102],
       [ 101, 2013, 1024, ..., 1998, 2017,  102],
       ...,
       [ 101, 2013, 1024, ...,    0,    0,    0],
       [ 101, 2013, 1024, ..., 1005, 1056,  102],
       [ 101, 2013, 1024, ..., 1998, 4983,  102]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(7919, 250), dtype=int32, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(7919, 250), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1]], dtype=int32)>}

In [ ]:
bert_clf.compile(optimizer=tf.compat.v1.train.AdamOptimizer(learning_rate=2e-5),
              loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
              metrics=[tf.keras.metrics.CategoricalAccuracy()])

# Train the model
history = bert_clf.fit(
    x=[bert_train['input_ids'],
       bert_train['attention_mask']],
    y=y_train_1_hot,
    epochs=5,
    batch_size=16,
    validation_data=([bert_val['input_ids'],
                      bert_val['attention_mask']],
                     y_val_1_hot),
    callbacks=[Metrics(([bert_val['input_ids'],
                         bert_val['attention_mask']],
                        y_val_1_hot))]
)

Epoch 1/5


2024-03-11 22:14:13.397136: W tensorflow/core/common_runtime/bfc_allocator.cc:479] Allocator (GPU_0_bfc) ran out of memory trying to allocate 46.88MiB (rounded to 49152000)requested by op tf_bert_for_sequence_classification/bert/encoder/layer_._5/intermediate/dense/Tensordot/MatMul
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2024-03-11 22:14:13.397203: I tensorflow/core/common_runtime/bfc_allocator.cc:1033] BFCAllocator dump for GPU_0_bfc
2024-03-11 22:14:13.397215: I tensorflow/core/common_runtime/bfc_allocator.cc:1040] Bin (256): 	Total Chunks: 105, Chunks in use: 103. 26.2KiB allocated for chunks. 25.8KiB in use in bin. 792B client-requested in use in bin.
2024-03-11 22:14:13.397219: I tensorflow/core/common_runtime/bfc_allocator.cc:1040] Bin (512): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in b

KeyboardInterrupt: 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# summarize history for accuracy
plt.plot(history.history['categorical_accuracy'])
plt.plot(history.history['val_categorical_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'dev'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'dev'], loc='upper right')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

predictions = np.argmax(tf.nn.softmax(bert_clf.predict([bert_test['input_ids'],
                                                        bert_test['attention_mask']]).logits, axis=-1),
                        -1)
print(classification_report(y_test, predictions,
                            target_names=twenty_train.target_names))

2. We can add more flexibility such as extra layers, pooling mechanisms and freezing the model by creating a custom Model.

In [ ]:
from transformers import TFAutoModel

bert_backbone = TFAutoModel.from_pretrained('bert-base-uncased',)
bert_backbone.summary()

In [ ]:
class BERTClassifier(tf.keras.Model):
  def __init__(self, bert, num_classes, freeze=False, apply_dropout=True):
    super().__init__()
    self.bert = bert
    self.apply_dropout = apply_dropout
    self.bert.trainable = True
    if freeze:
        self.bert.trainable = False
    self.pool = tf.keras.layers.GlobalMaxPooling1D(name='max_pool')
    self.dropout = tf.keras.layers.Dropout(0.5, name='dropout')
    self.clf = tf.keras.layers.Dense(num_classes, name='clf',
                                     activation='softmax')

  def call(self, inputs, training=None):
    # input_ids, attention_mask = inputs
    x = self.bert(inputs, training=training)
    x = self.pool(x.last_hidden_state)
    if self.apply_dropout:
        x = self.dropout(x, training=training)
    x = self.clf(x, training=training)
    return x

  def print_summary(self, line_length=None, positions=None, print_fn=None):
    # Fake forward pass to build graph
    x = np.zeros((1, 250), dtype=np.int32)
    self.predict([x, x])
    self.summary(line_length=line_length, positions=positions,
                 print_fn=print_fn)

In [ ]:
bert_classifier = BERTClassifier(bert_backbone,
                                 len(twenty_train.target_names),
                                 freeze=False, apply_dropout=True)
print(bert_classifier.print_summary())
bert_classifier.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
                        loss=tf.keras.losses.CategoricalCrossentropy(
                       from_logits=False
                       ),
                   metrics=[tf.keras.metrics.CategoricalAccuracy()])

bert_classifier.fit(x=[bert_train['input_ids'],
                  bert_train['attention_mask']
                  ],
                  y=y_train_1_hot, batch_size=16, verbose=1,
                  validation_data=([bert_val['input_ids'],
                                    bert_val['attention_mask']
                                    ], y_val_1_hot),
               shuffle=True,
               epochs=5,)